# Benchmark — Fase 1: eficiencia del detector/segmentador

## ¿Qué decide este notebook?

**Qué motor de detección/segmentación conviene** entre `sam3_text` (SAM3 por prompt de
texto) y `yolo_sam3` (YOLO localiza → SAM3 segmenta por box-prompt), midiendo **solo
eficiencia** (FPS y VRAM pico). No hay ground-truth, así que no medimos exactitud.

## ¿Por qué solo el detector, sin tracker?

El costo del pipeline lo **domina la detección** (~todo el cómputo por frame); el tracker
es casi gratis. Por eso el detector se elige por **eficiencia** y el tracker por
**consistencia** (eso es la Fase 2). Aquí corremos en `mode="segmentation"` **sin
tracker**: gracias a `detector_in_segmentation`, `yolo_sam3` ya funciona como
segmentador suelto. Sin tracker no hay `obj_id` estable, así que las métricas de
trayectoria/máscara **no aplican** (quedan N/A) — y no las necesitamos para decidir
eficiencia.

## ¿Por qué es honesto (sin fuga de datos)?

YOLO (`best.pt`) se entrenó en *fase_1* con los 103 videos NO-testing (splits 0+1,
anti-leak). Los 20 de **testing (`split=2`) quedaron intocados** → corremos sobre
testing y es justo para ambos detectores.

## Requisitos (correr EN EL POD, con GPU)

- Código al día en `develop`; si `import src.eval` falla, `pip install -e .`.
- `.env` con `CONFIG_FILENAME=01_yolo_sam3_config.json` (trae `yolo` y `botsort`).
- Pesos YOLO en `assets/yolo/best.pt`, SAM3 en `assets/sam3`, los 5 videos en `data/raw`.

Ejecutar las celdas de arriba a abajo.

## Paso 1 — Videos y detectores a evaluar

In [ ]:
from src.core.batch import run_batch
from src.eval.benchmark import aggregate_config, benchmark_videos, comparison_table

# Mismos 5 videos que la Fase 2 (seed=36: videos cortos, sin el de ~13 min que daba OOM).
videos = benchmark_videos(n=5, seed=36)
print("videos del benchmark (split=2, seed=36):")
for v in videos:
    print(" ", v)

# Fase 1: cada detector se mide SOLO (mode=segmentation, sin tracker).
DETECTORS = ["sam3_text", "yolo_sam3"]
print(f"\n{len(DETECTORS)} detectores a evaluar (solo eficiencia).")

## Paso 2 — Correr cada detector y medir

Por cada detector se llama a `run_batch` en `mode="segmentation"` con:

- **`sampling="auto"`** → cuota muestreada (memoria acotada; basta para medir FPS/VRAM
  por frame, que es una tasa).
- **`include_masks=False`** → eficiencia pura, JSON ligero.
- **`run_label=f"detectors/<det>"`** → cada detector escribe en su subcarpeta
  (`outputs/inference/detectors/<det>/…`), sin pisarse y con **skip-done por config**.
- **`progress=True`** → barra de progreso por video.

`aggregate_config` produce **una fila por detector**; aquí solo nos quedamos con
`fps`/`peak_vram_mb` (el resto es N/A sin tracker). La fila se **persiste al CSV** apenas
se calcula (resiliente a crash: `RESUME=True` reanuda los detectores que falten).

In [ ]:
import gc

import pandas as pd
import torch

from src.utils import PROJECT_ROOT

CSV_PATH = PROJECT_ROOT / "outputs" / "benchmark" / "detectors.csv"
RESUME = False  # True para reanudar tras un crash (salta los detectores ya guardados).

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if not RESUME and CSV_PATH.exists():
    CSV_PATH.unlink()
done = set(pd.read_csv(CSV_PATH)["config"]) if CSV_PATH.exists() else set()
if done:
    print("reanudando; ya guardados:", done)

for det in DETECTORS:
    label = f"seg+{det}"
    if label in done:
        print(f"skip {label} (ya en el CSV)")
        continue
    print(f"\n===== {label}  (segmentation) =====")
    summary = run_batch(
        mode="segmentation",
        videos=videos,
        detector=det,
        sampling="auto",  # cuota: memoria acotada
        include_masks=False,  # eficiencia: sin máscara
        render_video=False,
        overwrite=not RESUME,  # fresh run reprocesa; resume usa skip-done
        run_label=f"detectors/{det}",  # salidas aisladas por detector
        progress=True,
    )
    row = aggregate_config(label, summary)
    comparison_table([row]).to_csv(
        CSV_PATH, mode="a", header=not CSV_PATH.exists(), index=False
    )
    print("fila:", {k: row[k] for k in ("config", "fps", "peak_vram_mb")})

    del summary
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Paso 3 — Tabla de eficiencia

Sin tracker, **solo `fps` y `peak_vram_mb` son válidas**; las columnas de
trayectoria/máscara quedan N/A y se omiten.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
eff = df[["config", "fps", "peak_vram_mb"]]
print(f"Eficiencia de detectores ({len(df)}/{len(DETECTORS)}):")
eff

## Cómo leer y decidir

| Columna | Mejor cuando |
|---|---|
| `fps` | **mayor** (más rápido) |
| `peak_vram_mb` | **menor** (menos VRAM) |

- **Hipótesis:** `yolo_sam3` debería ser más rápido (YOLO filtra el espacio antes de
  SAM3). La tabla lo confirma o refuta con nuestros videos.
- **Decisión:** se elige el detector más eficiente para llevarlo a la **Fase 2**
  (trackers). Sin GTesta es la única métrica rigurosa; la exactitud queda para la
  evaluación con ground-truth (futuro).